# TextMamba3D AutoResearch - Layer 0

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

# Install with Drive pip cache (avoids recompilation)
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'

# Clone/pull with retry
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed: {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed after 3 attempts')
    os.chdir(REPO_DIR)
print(f'Repo: {os.getcwd()}')

# Data
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
cases = [d for d in os.listdir(DATA_DIR) if d.startswith('BraTS')]
print(f'Data: {len(cases)} cases')
print('Setup complete')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fri Mar 27 13:26:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|          

## L0-1: ET/WT ratio relabeling
Relabel ET as NCR when ET/WT volume ratio < threshold

In [2]:
# --- Eval: L0-1_et_wt_ratio_0.02 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-wt-ratio', '0.02'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-1_et_wt_ratio_0.02',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-wt-ratio 0.02
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8280, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8137, TC=0.8930, WT=0.8136) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6249 (ET=0.7707, TC=0.2924, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8614 (ET=0.7217, TC=0.9424, WT=0.9201) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6677, TC=0.8838, WT=0.7915) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  BraTS20_Training_107: Dice=0.9

In [3]:
# --- Eval: L0-1_et_wt_ratio_0.03 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-wt-ratio', '0.03'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-1_et_wt_ratio_0.03',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-wt-ratio 0.03
.00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.3681 (ET=0.0000, TC=0.2925, WT=0.8118) HD95_ET=nan
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8612 (ET=0.7215, TC=0.9421, WT=0.9201) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  BraTS20_Training_107: Dice=0.9

In [4]:
# --- Eval: L0-1_et_wt_ratio_0.04 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-wt-ratio', '0.04'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-1_et_wt_ratio_0.04',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-wt-ratio 0.04
.00
  BraTS20_Training_100: Dice=0.9086 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.3681 (ET=0.0000, TC=0.2927, WT=0.8118) HD95_ET=nan
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7217, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9504 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  BraTS20_Training_107: Dice=0.9

In [5]:
# --- Eval: L0-1_et_wt_ratio_0.05 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-wt-ratio', '0.05'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-1_et_wt_ratio_0.05',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-wt-ratio 0.05
.00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8280, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8137, TC=0.8930, WT=0.8136) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.3680 (ET=0.0000, TC=0.2924, WT=0.8118) HD95_ET=nan
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8614 (ET=0.7217, TC=0.9424, WT=0.9201) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6677, TC=0.8838, WT=0.7915) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  BraTS20_Training_107: Dice=0.9

## L0-2: Advanced PP parameter sweep
Sweep et_min_size and wt_min_size with best et_wt_ratio

In [6]:
# --- Eval: L0-2_et_min_size_50_wt_min_size_200 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '50', '--advanced-pp', '--wt-min-size', '200'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_50_wt_min_size_200',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 50 --advanced-pp --wt-min-size 200
00
  BraTS20_Training_100: Dice=0.9084 (ET=0.8280, TC=0.9583, WT=0.9388) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8134, TC=0.8930, WT=0.8137) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6252 (ET=0.7710, TC=0.2927, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8605 (ET=0.7215, TC=0.9422, WT=0.9176) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7809 (ET=0.6676, TC=0.8837, WT=0.7914) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  

In [7]:
# --- Eval: L0-2_et_min_size_50_wt_min_size_500 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '50', '--advanced-pp', '--wt-min-size', '500'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_50_wt_min_size_500',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 50 --advanced-pp --wt-min-size 500
00
  BraTS20_Training_100: Dice=0.9086 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6252 (ET=0.7710, TC=0.2927, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7217, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9504 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  

In [8]:
# --- Eval: L0-2_et_min_size_50_wt_min_size_1000 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '50', '--advanced-pp', '--wt-min-size', '1000'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_50_wt_min_size_1000',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 50 --advanced-pp --wt-min-size 1000
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8431 (ET=0.8134, TC=0.8930, WT=0.8228) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7216, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6678, TC=0.8839, WT=0.7914) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9341, TC=0.9578, WT=0.9595) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [9]:
# --- Eval: L0-2_et_min_size_100_wt_min_size_200 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '100', '--advanced-pp', '--wt-min-size', '200'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_100_wt_min_size_200',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 100 --advanced-pp --wt-min-size 200
00
  BraTS20_Training_100: Dice=0.9083 (ET=0.8279, TC=0.9583, WT=0.9388) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6252 (ET=0.7710, TC=0.2927, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7217, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9504 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [10]:
# --- Eval: L0-2_et_min_size_100_wt_min_size_500 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '100', '--advanced-pp', '--wt-min-size', '500'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_100_wt_min_size_500',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 100 --advanced-pp --wt-min-size 500
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8612 (ET=0.7215, TC=0.9421, WT=0.9201) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [11]:
# --- Eval: L0-2_et_min_size_100_wt_min_size_1000 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '100', '--advanced-pp', '--wt-min-size', '1000'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_100_wt_min_size_1000',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 100 --advanced-pp --wt-min-size 1000
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8431 (ET=0.8134, TC=0.8930, WT=0.8228) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7216, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6678, TC=0.8839, WT=0.7914) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9341, TC=0.9578, WT=0.9595) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41


In [12]:
# --- Eval: L0-2_et_min_size_200_wt_min_size_200 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '200', '--advanced-pp', '--wt-min-size', '200'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_200_wt_min_size_200',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 200 --advanced-pp --wt-min-size 200
00
  BraTS20_Training_100: Dice=0.9084 (ET=0.8280, TC=0.9583, WT=0.9388) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8137, TC=0.8930, WT=0.8136) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6249 (ET=0.7707, TC=0.2924, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8614 (ET=0.7217, TC=0.9424, WT=0.9201) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6677, TC=0.8838, WT=0.7915) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [13]:
# --- Eval: L0-2_et_min_size_200_wt_min_size_500 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '200', '--advanced-pp', '--wt-min-size', '500'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_200_wt_min_size_500',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 200 --advanced-pp --wt-min-size 500
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8612 (ET=0.7215, TC=0.9421, WT=0.9201) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [14]:
# --- Eval: L0-2_et_min_size_200_wt_min_size_1000 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '200', '--advanced-pp', '--wt-min-size', '1000'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_200_wt_min_size_1000',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 200 --advanced-pp --wt-min-size 1000
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8431 (ET=0.8134, TC=0.8930, WT=0.8228) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7216, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6678, TC=0.8839, WT=0.7914) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9341, TC=0.9578, WT=0.9595) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41


In [15]:
# --- Eval: L0-2_et_min_size_500_wt_min_size_200 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '500', '--advanced-pp', '--wt-min-size', '200'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_500_wt_min_size_200',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 500 --advanced-pp --wt-min-size 200
00
  BraTS20_Training_100: Dice=0.9084 (ET=0.8279, TC=0.9583, WT=0.9389) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8134, TC=0.8930, WT=0.8139) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7216, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7810 (ET=0.6678, TC=0.8839, WT=0.7914) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9341, TC=0.9578, WT=0.9595) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [16]:
# --- Eval: L0-2_et_min_size_500_wt_min_size_500 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '500', '--advanced-pp', '--wt-min-size', '500'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_500_wt_min_size_500',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 500 --advanced-pp --wt-min-size 500
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8280, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8134, TC=0.8930, WT=0.8137) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6252 (ET=0.7710, TC=0.2927, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8605 (ET=0.7215, TC=0.9422, WT=0.9176) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7501, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7809 (ET=0.6676, TC=0.8837, WT=0.7914) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
 

In [17]:
# --- Eval: L0-2_et_min_size_500_wt_min_size_1000 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp', '--et-min-size', '500', '--advanced-pp', '--wt-min-size', '1000'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-2_et_min_size_500_wt_min_size_1000',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp --et-min-size 500 --advanced-pp --wt-min-size 1000
00
  BraTS20_Training_100: Dice=0.9087 (ET=0.8280, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8430 (ET=0.8134, TC=0.8928, WT=0.8227) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6250 (ET=0.7707, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8854 (ET=0.8462, TC=0.9150, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8614 (ET=0.7217, TC=0.9423, WT=0.9202) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8690 (ET=0.7500, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6678, TC=0.8839, WT=0.7915) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41


## L0-3: TTA ablation
Compare with and without TTA (BraTS 2023 winner found TTA can hurt)

In [18]:
# --- Eval: L0-3_with_tta ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--advanced-pp'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-3_with_tta',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --advanced-pp
00
  BraTS20_Training_100: Dice=0.9086 (ET=0.8279, TC=0.9583, WT=0.9398) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8401 (ET=0.8136, TC=0.8930, WT=0.8138) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6252 (ET=0.7710, TC=0.2927, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8462, TC=0.9151, WT=0.8951) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8613 (ET=0.7217, TC=0.9423, WT=0.9200) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8692 (ET=0.7507, TC=0.9102, WT=0.9467) HD95_ET=4.36
  BraTS20_Training_021: Dice=0.7811 (ET=0.6679, TC=0.8839, WT=0.7916) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9504 (ET=0.9342, TC=0.9578, WT=0.9594) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8338 (ET=0.8230, TC=0.9398, WT=0.7386) HD95_ET=1.41
  BraTS20_Training_107: Dice=0.9277 (ET=0.9373, TC=

In [19]:
# --- Eval: L0-3_no_tta ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--advanced-pp'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-3_no_tta',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --advanced-pp

  BraTS20_Training_100: Dice=0.9067 (ET=0.8286, TC=0.9549, WT=0.9367) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8287 (ET=0.7987, TC=0.8814, WT=0.8061) HD95_ET=1.41
  BraTS20_Training_331: Dice=0.6338 (ET=0.7637, TC=0.3319, WT=0.8059) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8813 (ET=0.8372, TC=0.9136, WT=0.8929) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.8566 (ET=0.7172, TC=0.9385, WT=0.9142) HD95_ET=3.00
  BraTS20_Training_369: Dice=0.8723 (ET=0.7545, TC=0.9216, WT=0.9408) HD95_ET=4.12
  BraTS20_Training_021: Dice=0.7814 (ET=0.6750, TC=0.8819, WT=0.7874) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9461 (ET=0.9286, TC=0.9556, WT=0.9542) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8474 (ET=0.8678, TC=0.9404, WT=0.7341) HD95_ET=1.00
  BraTS20_Training_107: Dice=0.9218 (ET=0.9300, TC=0.9386, 

## L0-4: ET probability boost
Multiply ET softmax channel by boost factor before argmax

In [20]:
# --- Eval: L0-4_et_boost_1.0 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--et-boost', '1.0'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-4_et_boost_1.0',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --et-boost 1.0
0
  BraTS20_Training_100: Dice=0.9085 (ET=0.8284, TC=0.9582, WT=0.9388) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8394 (ET=0.8127, TC=0.8932, WT=0.8122) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6246 (ET=0.7695, TC=0.2925, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8853 (ET=0.8459, TC=0.9151, WT=0.8949) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.9132 (ET=0.8800, TC=0.9429, WT=0.9166) HD95_ET=1.00
  BraTS20_Training_369: Dice=0.9154 (ET=0.8894, TC=0.9101, WT=0.9467) HD95_ET=1.00
  BraTS20_Training_021: Dice=0.7824 (ET=0.6736, TC=0.8836, WT=0.7901) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9505 (ET=0.9341, TC=0.9577, WT=0.9598) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8495 (ET=0.8704, TC=0.9396, WT=0.7386) HD95_ET=1.00
  BraTS20_Training_107: Dice=0.9279 (ET=0.9372, TC=

In [21]:
# --- Eval: L0-4_et_boost_1.1 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--et-boost', '1.1'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-4_et_boost_1.1',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --et-boost 1.1
0
  BraTS20_Training_100: Dice=0.9073 (ET=0.8260, TC=0.9572, WT=0.9385) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8422 (ET=0.8183, TC=0.8957, WT=0.8125) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6274 (ET=0.7754, TC=0.2951, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8856 (ET=0.8475, TC=0.9145, WT=0.8946) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.9120 (ET=0.8777, TC=0.9417, WT=0.9165) HD95_ET=1.00
  BraTS20_Training_369: Dice=0.9138 (ET=0.8863, TC=0.9083, WT=0.9466) HD95_ET=1.00
  BraTS20_Training_021: Dice=0.7811 (ET=0.6700, TC=0.8831, WT=0.7900) HD95_ET=1.41
  BraTS20_Training_189: Dice=0.9508 (ET=0.9345, TC=0.9581, WT=0.9598) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8496 (ET=0.8705, TC=0.9398, WT=0.7387) HD95_ET=1.00
  BraTS20_Training_107: Dice=0.9267 (ET=0.9354, TC=

In [22]:
# --- Eval: L0-4_et_boost_1.2 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--et-boost', '1.2'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-4_et_boost_1.2',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --et-boost 1.2
0
  BraTS20_Training_100: Dice=0.9058 (ET=0.8228, TC=0.9563, WT=0.9382) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8442 (ET=0.8222, TC=0.8976, WT=0.8127) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6295 (ET=0.7797, TC=0.2972, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8855 (ET=0.8486, TC=0.9138, WT=0.8942) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.9102 (ET=0.8747, TC=0.9395, WT=0.9165) HD95_ET=1.00
  BraTS20_Training_369: Dice=0.9123 (ET=0.8835, TC=0.9068, WT=0.9466) HD95_ET=1.00
  BraTS20_Training_021: Dice=0.7802 (ET=0.6674, TC=0.8831, WT=0.7901) HD95_ET=1.73
  BraTS20_Training_189: Dice=0.9506 (ET=0.9339, TC=0.9582, WT=0.9598) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8497 (ET=0.8703, TC=0.9400, WT=0.7387) HD95_ET=1.00
  BraTS20_Training_107: Dice=0.9260 (ET=0.9345, TC=

In [23]:
# --- Eval: L0-4_et_boost_1.3 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--et-boost', '1.3'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-4_et_boost_1.3',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --et-boost 1.3
0
  BraTS20_Training_100: Dice=0.9044 (ET=0.8199, TC=0.9554, WT=0.9380) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8466 (ET=0.8271, TC=0.8998, WT=0.8128) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6308 (ET=0.7815, TC=0.2990, WT=0.8118) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8854 (ET=0.8491, TC=0.9131, WT=0.8939) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.9086 (ET=0.8714, TC=0.9379, WT=0.9166) HD95_ET=1.00
  BraTS20_Training_369: Dice=0.9110 (ET=0.8811, TC=0.9055, WT=0.9465) HD95_ET=1.00
  BraTS20_Training_021: Dice=0.7787 (ET=0.6629, TC=0.8830, WT=0.7901) HD95_ET=1.73
  BraTS20_Training_189: Dice=0.9510 (ET=0.9345, TC=0.9586, WT=0.9598) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8496 (ET=0.8698, TC=0.9403, WT=0.7388) HD95_ET=1.00
  BraTS20_Training_107: Dice=0.9247 (ET=0.9325, TC=

In [24]:
# --- Eval: L0-4_et_boost_1.5 ---
import subprocess, json, re, os, pathlib

DRIVE_CKPT = '/content/drive/MyDrive/TextMamba3D/checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'
REPO_DIR = '/content/TextMamba3D'

ckpt_path = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.isfile(ckpt_path), f'Checkpoint not found: {{ckpt_path}}'

cmd = [
    'python', 'evaluate_full.py',
    '--config', 'configs/archive/textbrats_a100_v5.yaml',
    '--checkpoint', ckpt_path,
    '--split', 'test',
    '--use-text', '--tta', '--et-boost', '1.5'
]
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
output = result.stdout
print(output[-1500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Parse metrics
metrics = {}
for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
            'hd95_ET', 'hd95_TC', 'hd95_WT']:
    m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', output)
    if m:
        metrics[key] = float(m.group(1))
        metrics[key + '_std'] = float(m.group(2))

record = {
    'experiment_id': 'L0-4_et_boost_1.5',
    'checkpoint': 'best_v5.0.pth',
    'config': 'configs/archive/textbrats_a100_v5.yaml',
    'split': 'test',
    'metrics': metrics,
    'returncode': result.returncode,
}

# Append to Drive results file
if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        all_results = json.load(f)
else:
    all_results = []
all_results.append(record)
with open(DRIVE_RESULTS, 'w') as f:
    json.dump(all_results, f, indent=2)

print()
print('Metrics:', metrics)


Running: python evaluate_full.py --config configs/archive/textbrats_a100_v5.yaml --checkpoint /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth --split test --use-text --tta --et-boost 1.5
0
  BraTS20_Training_100: Dice=0.9021 (ET=0.8148, TC=0.9539, WT=0.9375) HD95_ET=1.00
  BraTS20_Training_088: Dice=0.8497 (ET=0.8334, TC=0.9026, WT=0.8130) HD95_ET=1.00
  BraTS20_Training_331: Dice=0.6331 (ET=0.7858, TC=0.3017, WT=0.8117) HD95_ET=1.41
  BraTS20_Training_215: Dice=0.8853 (ET=0.8507, TC=0.9117, WT=0.8933) HD95_ET=1.00
  BraTS20_Training_122: Dice=0.9055 (ET=0.8654, TC=0.9347, WT=0.9164) HD95_ET=1.00
  BraTS20_Training_369: Dice=0.9084 (ET=0.8760, TC=0.9027, WT=0.9465) HD95_ET=1.41
  BraTS20_Training_021: Dice=0.7765 (ET=0.6569, TC=0.8825, WT=0.7901) HD95_ET=1.73
  BraTS20_Training_189: Dice=0.9509 (ET=0.9341, TC=0.9588, WT=0.9598) HD95_ET=1.00
  BraTS20_Training_072: Dice=0.8497 (ET=0.8696, TC=0.9407, WT=0.7389) HD95_ET=1.00
  BraTS20_Training_107: Dice=0.9230 (ET=0.9297, TC=

## Results Summary

In [25]:
# ===== Summary =====
import json, os
DRIVE_RESULTS = '/content/drive/MyDrive/TextMamba3D/autoresearch_results.json'

if os.path.exists(DRIVE_RESULTS):
    with open(DRIVE_RESULTS) as f:
        results = json.load(f)
    print(f'Total experiments: {len(results)}')
    print()
    # Sort by dice_mean descending
    ranked = sorted(
        [r for r in results if r.get('metrics', {}).get('dice_mean')],
        key=lambda r: r['metrics']['dice_mean'],
        reverse=True,
    )
    print('Top results:')
    for i, r in enumerate(ranked[:10]):
        m = r['metrics']
        print(
            f"  {i+1}. {r['experiment_id']}: "
            f"dice_mean={m['dice_mean']:.4f}  "
            f"dice_ET={m.get('dice_ET', 0):.4f}"
        )
else:
    print('No results file found.')


Total experiments: 23

Top results:
  1. L0-2_et_min_size_100_wt_min_size_1000: dice_mean=0.8479  dice_ET=0.7896
  2. L0-2_et_min_size_100_wt_min_size_200: dice_mean=0.8475  dice_ET=0.7893
  3. L0-2_et_min_size_100_wt_min_size_500: dice_mean=0.8475  dice_ET=0.7893
  4. L0-4_et_boost_1.0: dice_mean=0.8474  dice_ET=0.7897
  5. L0-2_et_min_size_200_wt_min_size_1000: dice_mean=0.8469  dice_ET=0.7874
  6. L0-4_et_boost_1.1: dice_mean=0.8469  dice_ET=0.7889
  7. L0-2_et_min_size_200_wt_min_size_200: dice_mean=0.8466  dice_ET=0.7871
  8. L0-2_et_min_size_200_wt_min_size_500: dice_mean=0.8466  dice_ET=0.7871
  9. L0-4_et_boost_1.2: dice_mean=0.8465  dice_ET=0.7883
  10. L0-4_et_boost_1.3: dice_mean=0.8459  dice_ET=0.7872
